In [1]:
import pandas as pd
from Bio import PDB
import os, sys
import matplotlib.pyplot as plt

sys.path.insert(0,'..')
from functions.myfunctions import *
from functions.sequencefunctions import get_vregion_seq_with_gaps
import functions.distancefunctions as df

In [ ]:
structures_dir = '../data/structures/stcrdab_all_structures/imgt/'
datalist = pd.read_csv('../data/tcrdbset_unique_receptors.csv', index_col=0).drop_duplicates(subset = ['pdb'], keep='first')
datalist_bound = pd.read_csv('../data/tcrdbset_unique_receptors_bound.csv', index_col=0).drop_duplicates(subset = ['pdb'], keep='first')
datalist_unbound = pd.read_csv('../data/tcrdbset_unique_receptors_unbound.csv', index_col=0).drop_duplicates(subset = ['pdb'], keep='first')

datalist1 = pd.read_csv('../data/tcrdb_export_cleaned_withseqs.csv', index_col=0).drop_duplicates(subset = ['pdb'], keep='first')

In [3]:
datalist

,alpha_aa_imgt_shorter,beta_aa_imgt_shorter,pdb,Bchain,Achain,model,antigen_chain,antigen_type,antigen_name,antigen_het_name,...,affinity_method,affinity_temperature,affinity_pmid,engineered,alpha_aa_imgt,alpha_aa_imgt_withGaps,beta_aa_imgt,beta_aa_imgt_withGaps,epitope_aa,bound
0,AKTTQPDSMESTEGETVHLPCSHATISGNEYIYWYRQVPLQGPEYV...,AVTQSPRNKVTVTGGNVTLSCRQTNSHNYMYWYRQDTGHGLRLIHY...,['6l9l'],['H'],['G'],[0],['F'],['peptide'],['ser-pro-ser-tyr-ala-tyr-his-gln-phe'],['NA'],...,['NA'],['NA'],['NA'],[True],['AKTTQPDSMESTEGETVHLPCSHATISGNEYIYWYRQVPLQGPE...,['-AKTTQ-PDSMESTEGETVHLPCSHATISG-----NEYIYWYRQ...,['AVTQSPRNKVTVTGGNVTLSCRQTNSHNYMYWYRQDTGHGLRLI...,['--AVTQSPRNKVTVTGGNVTLSCRQTNSH-------NYMYWYRQ...,['SPSYAYHQF'],[True]
1,AKTTQPISMDSYEGQEVNITCNHNDIATSDYIMWYQQFPNQGPRFI...,DTAVSQTPKYLVRQTGKNESLKCEQNLGHNAMYWYKQDSKKLLKIM...,['7byd'],['J'],['I'],[0],['H'],['peptide'],['gly-gly-ala-ile'],['NA'],...,['NA'],['NA'],['NA'],[True],['AKTTQPISMDSYEGQEVNITCNHNDIATSDYIMWYQQFPNQGPR...,['-AKTTQ-PISMDSYEGQEVNITCNHNDIAT-----SDYIMWYQQ...,['DTAVSQTPKYLVRQTGKNESLKCEQNLGHNAMYWYKQDSKKLLK...,['DTAVSQTPKYLVRQTGKNESLKCEQNLGH-------NAMYWYKQ...,['GGAI'],[True]
2,AKTTQPTSMDCAEGRAANLPCNHSTISGNEYVYWYRQIHSQGPQYI...,TGVSQNPRHKITKRGQNVTFRCDPISEHNRLYWYRQTLGQGPEFLT...,"['3dx9', '3dxa']","['D', 'E']","['C', 'D']","[0, 0]","['NA', 'C']","['NA', 'peptide']","['NA', 'ebv decapeptide epitope']","['NA', 'NA']",...,"['NA', 'NA']","['NA', 'NA']","['NA', 'NA']","[True, True]",['AKTTQPTSMDCAEGRAANLPCNHSTISGNEYVYWYRQIHSQGPQ...,['-AKTTQ-PTSMDCAEGRAANLPCNHSTISG-----NEYVYWYRQ...,['TGVSQNPRHKITKRGQNVTFRCDPISEHNRLYWYRQTLGQGPEF...,['-TGVSQNPRHKITKRGQNVTFRCDPISEH-------NRLYWYRQ...,"['NA', 'EENLLDFVRF']","[False, True]"
3,AQEVTQIPAALSVPEGENLVLNCSFTDSAIYNLQWFRQDPGKGLTS...,NAGVTQTPKFQVLKTGQSMTLQCSQDMNHEYMSWYRQDPGMGLRLI...,"['6zkw', '6zkx']","['E', 'E']","['D', 'D']","[0, 0]","['C', 'C']","['peptide', 'peptide']",['enoyl-[acyl-carrier-protein] reductase [nadh...,"['NA', 'NA']",...,"['NA', 'NA']","['NA', 'NA']","['NA', 'NA']","[True, True]",['AQEVTQIPAALSVPEGENLVLNCSFTDSAIYNLQWFRQDPGKGL...,['AQEVTQIPAALSVPEGENLVLNCSFTDSA------IYNLQWFRQ...,['NAGVTQTPKFQVLKTGQSMTLQCSQDMNHEYMSWYRQDPGMGLR...,['NAGVTQTPKFQVLKTGQSMTLQCSQDMNH-------EYMSWYRQ...,"['RLPAKAPLL', 'RLPAKAPLLGCG']","[True, True]"
4,AQEVTQIPAALSVPEGENLVLNCSFTDSAIYNLQWFRQDPGKGLTS...,AGVTQTPRYLIKTRGQQVTLSCSPISGHRSVSWYQQTPGQGLQFLF...,"['5brz', '5bs0']","['E', 'E']","['D', 'D']","[0, 0]","['C', 'C']","['peptide', 'peptide']","['glu-val-asp-pro-ile-gly-his-leu-tyr', 'titin']","['NA', 'NA']",...,"['NA', 'NA']","['NA', 'NA']","['NA', 'NA']","[True, True]",['AQEVTQIPAALSVPEGENLVLNCSFTDSAIYNLQWFRQDPGKGL...,['AQEVTQIPAALSVPEGENLVLNCSFTDSA------IYNLQWFRQ...,['AGVTQTPRYLIKTRGQQVTLSCSPISGHRSVSWYQQTPGQGLQF...,['-AGVTQTPRYLIKTRGQQVTLSCSPISGH-------RSVSWYQQ...,"['EVDPIGHLY', 'ESDPIVAQY']","[True, True]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203,SQQGEEDPQALSIQEGENATMNCSYKTSINNLQWYRQNSGRGLVHL...,NAGVTQTPKFRVLKTGQSMTLLCAQDMNHEYMYWYRQDPGMGLRLI...,['7t2b'],['E'],['D'],[0],['C'],['peptide'],['pneumolysin-derived peptide'],['NA'],...,['NA'],['NA'],['NA'],[True],['SQQGEEDPQALSIQEGENATMNCSYKTSINNLQWYRQNSGRGLV...,['SQQGEEDPQALSIQEGENATMNCSYKTSI-------NNLQWYRQ...,['NAGVTQTPKFRVLKTGQSMTLLCAQDMNHEYMYWYRQDPGMGLR...,['NAGVTQTPKFRVLKTGQSMTLLCAQDMNH-------EYMYWYRQ...,['GATGLAWEWWRTVYE'],[True]
204,SQQGEEDPQALSIQEGENATMNCSYKTSINNLQWYRQNSGRGLVHL...,GAVVSQHPSWVISKSGTSVKIECRSLDFQATTMFWYRQFPKQSLML...,['1ymm'],['E'],['D'],[0],['C'],['peptide'],['mbp peptide'],['NA'],...,['NA'],['NA'],['NA'],[True],['SQQGEEDPQALSIQEGENATMNCSYKTSINNLQWYRQNSGRGLV...,['SQQGEEDPQALSIQEGENATMNCSYKTSI-------NNLQWYRQ...,['GAVVSQHPSWVISKSGTSVKIECRSLDFQATTMFWYRQFPKQSL...,['GAVVSQHPSWVISKSGTSVKIECRSLDFQ------ATTMFWYRQ...,['ENPVVHFFKNIVTP'],[True]
205,SVTQPDARVTVSEGASLQLRCKYSYSATPYLFWYVQYPRQGPQLLL...,AAVTQSPRNKVAVTGEKVTLSCNQTNNHNNMYWYRQDTGHELRLIY...,['3e3q'],['a'],['Z'],[0],['b'],['peptide'],['unknown'],['NA'],...,['NA'],['NA'],['NA'],[True],['SVTQPDARVTVS

In [4]:
p = PDB.PDBParser()

distances_a = {}
distances_b = {}

unique_pdbs = sorted(datalist1['pdb'].tolist())
print(len(unique_pdbs))

for i, ppdb in enumerate(unique_pdbs):
        print(ppdb, i, 'of', len(unique_pdbs))
        ppdb = ppdb.strip('[]\' ')
        s = ppdb + '.pdb'
        print(ppdb)

        struc = p.get_structure(ppdb, structures_dir + s)
        achain = list(datalist1.loc[datalist1.pdb == ppdb]['Achain'])[0]
        bchain = list(datalist1.loc[datalist1.pdb == ppdb]['Bchain'])[0]

        da = df.mindist_pairwise_distances(struc[0][achain],struc[0][achain])
        db = df.mindist_pairwise_distances(struc[0][bchain],struc[0][bchain])
        distances_a[ppdb] = da
        distances_b[ppdb] = db

340
1ao7 0 of 340
1ao7
1bd2 1 of 340
1bd2
1d9k 2 of 340
1d9k
1fo0 3 of 340
1fo0
1fyt 4 of 340
1fyt
1g6r 5 of 340
1g6r
1j8h 6 of 340
1j8h
1kgc 7 of 340
1kgc
1kj2 8 of 340
1kj2
1mi5 9 of 340
1mi5
1mwa 10 of 340
1mwa
1nam 11 of 340
1nam
1oga 12 of 340
1oga
1qrn 13 of 340
1qrn
1qse 14 of 340
1qse
1qsf 15 of 340
1qsf
1tcr 16 of 340
1tcr
1u3h 17 of 340
1u3h
1ymm 18 of 340
1ymm
1zgl 19 of 340
1zgl
2ak4 20 of 340
2ak4
2bnq 21 of 340
2bnq
2bnr 22 of 340
2bnr
2bnu 23 of 340
2bnu
2cde 24 of 340
2cde
2cdf 25 of 340
2cdf
2cdg 26 of 340
2cdg
2ckb 27 of 340
2ckb
2e7l 28 of 340
2e7l
2esv 29 of 340
2esv
2eyr 30 of 340
2eyr
2eys 31 of 340
2eys
2eyt 32 of 340
2eyt
2f53 33 of 340
2f53
2f54 34 of 340
2f54
2gj6 35 of 340
2gj6
2ial 36 of 340
2ial
2iam 37 of 340
2iam
2ian 38 of 340
2ian
2nw2 39 of 340
2nw2
2nx5 40 of 340
2nx5
2oi9 41 of 340
2oi9
2ol3 42 of 340
2ol3
2p5e 43 of 340
2p5e
2p5w 44 of 340
2p5w
2pxy 45 of 340
2pxy
2pye 46 of 340
2pye
2pyf 47 of 340
2pyf
2q86 48 of 340
2q86
2vlj 49 of 340
2vlj
2vlk 5

In [ ]:
import pickle

outfolder = '../data/outputs/'

with open(outfolder + 'intrachain_a_mindists.pickle', 'wb') as handle:
    pickle.dump(distances_a, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open(outfolder + 'intrachain_b_mindists.pickle', 'wb') as handle:
    pickle.dump(distances_b, handle, protocol=pickle.HIGHEST_PROTOCOL)